# 📖 中文小说模型对比测试（novel-model-benchmark）

在 ModelScope 免费 GPU（NVIDIA A10 24GB）上，用同一段剧情测试两个 27B 中文小说模型的写正文能力。

**怎么用（就这么简单）：**
1. 菜单栏 **Run → Run All Cells（运行全部）**，然后回到这里往下看；
2. 运行到「第 5 步」会自动下载并加载默认模型①（首次约 16.5GB、10~40 分钟，**下载过就永久保存，之后秒加载**）；
3. 到「第 6 步」在输入框里**粘贴你的剧情**，点【✍️ 开始生成】；
4. 正文逐字显示，并**自动保存**到 `/mnt/workspace/novel-model-benchmark/results/`。

**两个模型：**

| # | 模型 | 运行方式 | 下载体积 | A10 24GB |
|---|------|---------|---------|----------|
| ① | WebNovel Writer（西幻·白描风） | llama.cpp + Q4_K_M 量化 | ≈16.5GB | ✅ 推荐 |
| ② | 玄幻 DPO（结构化大纲→正文） | 4bit 底模 + LoRA 适配器 | ≈23.4GB | ⚠️ 尝试性支持 |

> ⚠️ 模型②的 4bit 底模（22.3GB）已接近 A10 整卡显存（24GB），属于**尝试性支持**：万一显存不够，程序会自动按预案处理并给出明确结论（建议换 80GB 显卡的实例），**不需要你排查任何报错**。
>
> 第一次使用请先看仓库 README：注册 ModelScope → 打开映射链接 → 选 A10 GPU → 运行全部。遇到红色报错不要慌，截图后按 README「常见问题」处理。

## 第 1 步 · 配置（整个项目唯一可能需要看的代码格）

下面这个 Cell 集中了**全部**模型 ID、存储路径和默认参数，且默认值已经按两位模型作者的官方说明填好——**直接用即可，什么都不用改**。

只有两种情况才需要动它：换模型/换量化版本，或微调生成参数。其他 Cell 都是自动流程，请勿修改。

In [ ]:
# ============================================================
# 第 1 步：全局配置（唯一需要看的 Cell，默认值已按模型作者官方说明填好）
# ============================================================
CONFIG = {
    # ---------- 存储位置（ModelScope 免费实例自带 100GB 持久盘，关机不丢） ----------
    "workspace_root": "/mnt/workspace",
    # 模型权重 + 下载缓存统一放这里（只保留一份，避免双份缓存吃掉磁盘）
    "models_root": "/mnt/workspace/models",
    # Hugging Face 下载缓存目录（与模型同盘）
    "hf_cache": "/mnt/workspace/models/hf-cache",
    # 生成结果保存目录
    "results_dir": "/mnt/workspace/novel-model-benchmark/results",
    # 批量盲评模式的提示词文件夹（Notebook 同级的 prompts/）
    "prompts_dir": "prompts",

    # ---------- 下载源（按优先级自动切换） ----------
    "hf_official": "https://huggingface.co",  # 官方源，优先
    "hf_mirror":   "https://hf-mirror.com",   # 第三方镜像（非官方，仅在官方源失败时自动使用，不发送任何令牌）

    # ---------- 两个模型 ----------
    "models": {
        # 模型①：西幻网文写作模型（GGUF 量化，llama.cpp 运行）
        "webnovel": {
            "label": "① WebNovel Writer · 西幻白描风（约16.5GB，A10 首推）",
            "type": "gguf",
            "repo_id": "wcn123/Qwen3.5-27B-WebNovel-Writer-zh-GGUF",
            "gguf_file": "Qwen3.5-27B-WebNovel-Writer-zh-Q4_K_M.gguf",  # 仓库内唯一量化版
            "context": 4096,          # 上下文长度（默认 4096）
            # ↓ 生成参数 = 模型卡「使用建议」的官方推荐值
            "temperature": 0.7, "top_p": 0.90, "repeat_penalty": 1.10,
            # ↓ System prompt = 模型卡官方原文
            "system_prompt": "你是一位中文西幻网文写作助手，擅长创作高质量的小说正文。请根据用户的指令完成写作任务。",
            "max_new_tokens": 1700,   # 生成上限（防写不停）；实际篇幅由提示词里的字数目标控制
        },
        # 模型②：玄幻小说 DPO 模型（LoRA 适配器 + 4bit 底模，Transformers 运行）
        "xuanhuan": {
            "label": "② 玄幻 DPO · 结构化大纲转正文（底模约22.4GB，A10 尝试性支持）",
            "type": "peft4bit",
            "adapter_repo": "JiangLing-js/Qwen3.8-27B-Chinese-Xuanhuan-Novel-Writer-DPO",
            "base_repo": "unsloth/Qwen3.8-27B-unsloth-bnb-4bit",  # 官方 4bit 底模（22.4GB，免下 53GB 原始权重）
            "context": 4096,          # 默认上下文；OOM 时按预案只做一次降级
            "context_fallback": 2048, # 唯一一次降级尝试使用的上下文（只为 KV 缓存留余量，权重体积不变）
            "offload_gib": 20,        # 唯一一次降级尝试限制 GPU 占 20GiB，其余层自动放 CPU
            "min_speed_tps": 2.0,     # 实测低于此速度（token/秒）判定"不可用"，直接建议换 80GB 卡
            # ↓ 生成参数 = 模型作者 held-out 测试原参数（temperature/top_p/top_k/重复惩罚）
            "temperature": 0.85, "top_p": 0.90, "top_k": 40, "repetition_penalty": 1.05,
            "system_prompt": "你是一位中文玄幻小说写作助手。请根据提供的结构化信息直接写出连贯的小说场景正文，不要解释写作过程。",
            "max_new_tokens": 1700,
            "disable_thinking": True, # 训练时即关闭思考模式，推理必须同样关闭
        },
    },
    "default_model": "webnovel",   # 「运行全部」时自动加载模型①（首次约16.5GB）

    # ---------- 通用 ----------
    "target_chars": "500~1000",    # 提示词中声明的正文篇幅目标（中文字）
}

# 环境变量必须在 import huggingface 之前设置：把缓存统一指到持久盘，只留一份权重
import os
os.environ["HF_HOME"] = CONFIG["hf_cache"]
for d in (CONFIG["models_root"], CONFIG["results_dir"]):
    os.makedirs(d, exist_ok=True)

SHORT_NAME = {"webnovel": "webnovel-writer", "xuanhuan": "xuanhuan-dpo"}

print("✅ 配置加载完成")
print(f"   模型缓存目录 : {CONFIG['models_root']}")
print(f"   结果保存目录 : {CONFIG['results_dir']}")
print(f"   默认模型     : {CONFIG['default_model']}（{CONFIG['models'][CONFIG['default_model']]['label']}）")

## 第 2 步 · 环境自检（自动）

下载大文件之前，先确认这台云主机有 GPU、磁盘和内存够用。任何一项不满足会直接用中文告诉你怎么办。

In [ ]:
# ============================================================
# 第 2 步：环境自检（GPU / 磁盘 / 内存），有问题直接给人话提示
# ============================================================
import shutil, subprocess

def _run(cmd):
    """执行 shell 命令，返回 stdout（失败返回空字符串，不抛异常）。"""
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return ""

problems = []

# ---------- 1) GPU：需要 NVIDIA（A10 24GB） ----------
smi = _run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if not smi.strip():
    print("❌ 没有检测到 NVIDIA GPU。")
    print("   解决：ModelScope 网页 → 我的Notebook → 停止当前实例 → 重新启动时选择 GPU 镜像（NVIDIA A10 24GB）。")
    problems.append("无GPU")
else:
    first = smi.splitlines()[0]
    name, vram = [x.strip() for x in first.split(",")[:2]]
    vram_gib = float(vram.replace("MiB", "")) / 1024
    print(f"✅ GPU：{name}｜显存 {vram_gib:.1f} GiB")
    if vram_gib < 22:
        print("   ⚠️ 显存不足 22GiB：两个 27B 模型都跑不动，请换 A10(24GB) 或更大的卡。")
        problems.append("显存不足")

# ---------- 2) 磁盘：两个模型合计约 40GB（持久盘共 100GB） ----------
if os.path.exists(CONFIG["workspace_root"]):
    total, used, free = shutil.disk_usage(CONFIG["workspace_root"])
    print(f"✅ 磁盘：{CONFIG['workspace_root']} 剩余 {free/1000**3:.0f}GB / 共 {total/1000**3:.0f}GB")
    if free < 20 * 1000**3:
        print("   ⚠️ 剩余不足 20GB：先只测模型①；需要测模型②前请清理 /mnt/workspace。")
        problems.append("磁盘紧张")
else:
    print(f"⚠️ 未找到 {CONFIG['workspace_root']}（本机不是 ModelScope 环境？云端会自动出现）")

# ---------- 3) 内存（CPU 卸载时要用） ----------
freeg = _run("free -g")
try:
    avail = int([l for l in freeg.splitlines() if l.startswith("Mem:")][0].split()[6])
    print(f"✅ 内存：可用约 {avail}GB")
except Exception:
    pass

if problems:
    print(f"\n🔴 自检未通过（{problems}）。请先按上面的提示解决，再重新运行本 Cell。")
    raise SystemExit("环境自检未通过")
print("\n🟢 环境自检通过，请继续运行下一个 Cell。")

## 第 3 步 · 自动安装基础依赖（1~3 分钟）

缺什么装什么；国内网络会自动换清华镜像源。这里只装轻量通用包——**模型②专用的深度学习包等选到它时再装**，保证第一次能最快跑通模型①。

In [ ]:
# ============================================================
# 第 3 步：依赖自动安装（先查缺，再安装；默认源失败自动换清华镜像）
# ============================================================
import importlib, subprocess, sys

def pip_install(pkgs):
    """安装缺失的 pip 包：已装的跳过；默认源失败自动换清华镜像。"""
    need = []
    for p in pkgs:
        mod = p.split("[")[0].replace("-", "_")
        try:
            importlib.import_module(mod)
        except ImportError:
            need.append(p)
    if not need:
        print("✅ 依赖已就绪：" + ", ".join(pkgs))
        return
    print(f"⏳ 安装缺失依赖：{need} …")
    last_err = ""
    for extra in ([], ["-i", "https://pypi.tuna.tsinghua.edu.cn/simple"]):
        cmd = [sys.executable, "-m", "pip", "install", "-q", *extra, *need]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            break
        last_err = r.stderr[-600:]
    else:
        raise RuntimeError(f"依赖安装失败，请把下方信息截图求助：\n{last_err}")
    print("✅ 安装完成")

# 基础包：下载器 + 交互控件（深度学习栈在第 5 步选模型②时才装）
pip_install(["huggingface_hub", "ipywidgets"])

## 第 4 步 · 下载与通用工具（自动）

这一格包含：模型存在性检查、**断点续传下载**（中断后重跑会接着上次的进度继续）、官方源失败自动切第三方镜像、显存监控、结果自动保存、两个模型各自的提示词模板。

In [ ]:
# ============================================================
# 第 4 步：下载与通用工具
#   · smart_download : 先查模型是否存在 → 官方源下载(断点续传) → 失败自动切 hf-mirror
#   · show_gpu       : 显存监控
#   · save_result    : 结果自动保存（文件名 = 模型名_日期时间_测试编号）
#   · build_user_message : 按各模型作者推荐的输入模板包装你粘贴的剧情
# ============================================================
import os, re, gc, time, json, glob, shutil, datetime

STATE = {"loaded_model": None, "runner": None}   # 当前已加载的模型与运行方式

def _model_exists(repo_id, endpoint):
    """在指定下载源上检查模型是否存在（连不上视为不存在，自动换源）。"""
    try:
        from huggingface_hub import HfApi
        return HfApi(endpoint=endpoint).model_info(repo_id, timeout=20) is not None
    except Exception:
        return False

def smart_download(repo_id, filename=None):
    """下载模型（带断点续传与镜像降级）：
       ① 官方源 huggingface.co（中断后重跑本函数自动续传）
       ② 第三方镜像 hf-mirror.com（非官方节点，同样支持续传；强制匿名，绝不向镜像发送令牌）
       实现说明：hub 1.x 的下载端点在 import 时固化，因此每个源用 HfApi(endpoint=...) 独立客户端。
       两者都失败才报错，并给出可手动执行的补救命令。"""
    from huggingface_hub import HfApi
    for name, ep, token in [("官方源", CONFIG["hf_official"], None),
                            ("第三方镜像", CONFIG["hf_mirror"], False)]:
        if not _model_exists(repo_id, ep):
            print(f"   · {name}({ep}) 上未找到 {repo_id}（或网络不通），换下一个源…")
            continue
        try:
            what = filename or "整个仓库"
            print(f"⏳ 从{name}下载 {repo_id} / {what}")
            print("   （已下载过则直接复用；下载中断后重新运行本 Cell 会从断点继续）")
            api = HfApi(endpoint=ep, token=token)   # token=False：镜像强制匿名
            if filename:
                path = api.hf_hub_download(repo_id=repo_id, filename=filename)
            else:
                path = api.snapshot_download(repo_id=repo_id)
            print(f"✅ 已就绪：{path}")
            return path
        except Exception as e:
            print(f"   · {name}下载失败：{str(e)[:200]}")
    raise RuntimeError(
        f"\n🔴 两个下载源都失败了。补救方法（按顺序试）：\n"
        f"   1) 重新运行本 Cell（网络波动常常自愈，且会断点续传，不会重头下载）\n"
        f"   2) 仍失败：菜单 File → New → Terminal，执行下面两行后回来重跑本 Cell：\n"
        f"        export HF_ENDPOINT={CONFIG['hf_mirror']}\n"
        f"        （然后回到 Notebook 重新运行第 5 步的加载 Cell）\n"
        f"   3) 还不行：截图报错信息求助。")

def show_gpu(tag="当前"):
    out = _run("nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader").strip()
    print(f"🖥️  {tag}显存（已用/总量/利用率）：{out or '未知'}")

def _repo_line(k):
    c = CONFIG["models"][k]
    if c["type"] == "gguf":
        return f"{c['repo_id']} · {c['gguf_file']}"
    return f"{c['adapter_repo']} + 4bit底模 {c['base_repo']}"

def next_test_number(model_key):
    """扫描 results 目录，算出该模型的下一个测试编号（001 起）。"""
    pat = re.compile(re.escape(SHORT_NAME[model_key]) + r"_\d{8}_\d{6}_test(\d+)\.")
    nums = []
    for f in glob.glob(os.path.join(CONFIG["results_dir"], "*.txt")):
        m = pat.match(os.path.basename(f))
        if m:
            nums.append(int(m.group(1)))
    return max(nums, default=0) + 1

def save_result(model_key, task_label, user_text, gen_text, meta=None):
    """自动保存：.txt 纯正文 + .md 带元信息（模型/参数/耗时/显存/提示词）。文件名含模型名、时间、测试编号。"""
    meta = meta or {}
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    n = next_test_number(model_key)
    stem = f"{SHORT_NAME[model_key]}_{ts}_test{n:03d}"
    txt_path = os.path.join(CONFIG["results_dir"], stem + ".txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(gen_text.strip() + "\n")
    c = CONFIG["models"][model_key]
    params = {k: c.get(k) for k in ("temperature", "top_p", "top_k", "repeat_penalty",
                                    "repetition_penalty", "max_new_tokens") if k in c}
    params["context"] = meta.get("context", c.get("context"))
    md_path = txt_path.replace(".txt", ".md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write("# 生成记录\n\n"
                f"- 模型：{_repo_line(model_key)}\n"
                f"- 任务类型：{task_label}\n- 测试编号：{n:03d}\n- 时间：{ts}\n"
                f"- 生成参数：{json.dumps(params, ensure_ascii=False)}\n"
                f"- 耗时：{meta.get('elapsed', '?')} 秒｜速度：{meta.get('speed', '?')}\n"
                f"- 显存：{meta.get('vram', '?')}\n\n"
                f"## 提示词\n\n```\n{user_text.strip()}\n```\n\n## 正文\n\n{gen_text.strip()}\n")
    print(f"💾 已自动保存：\n   {txt_path}\n   {md_path}")
    return txt_path, md_path

# ---------- 提示词模板（均来自模型作者官方说明） ----------
# 模型①：模型卡「使用建议」里的 4 种任务格式，{text} 处填你粘贴的内容
WEBNOVEL_TASKS = {
    "骨架扩写": ("请将下面文本增强为更自然的小说正文。\n\n"
               "输入类型：事件骨架\n增强强度：高（从纯骨架到完整场景）\n长度目标：3.5x~6.0x\n"
               "重点：从骨架扩写完整场景：叙事结构、节奏铺排、感官填充、对话还原\n"
               "要求：严格保留骨架中的全部事实、人名、地名、术语、事件顺序与结果。不新增原文没有的信息。\n\n"
               "文本：\n{text}"),
    "场景扩写": ("任务：场景扩写\n场景描述：{text}\n"
               "要求：西幻风格，注重场景感和节奏，保持人物行为合理。\n\n请写出完整的小说场景。"),
    "正文增强": ("任务：正文增强\n目标：让这段更像成熟作者写出的西幻正文\n输入类型：场景beat\n"
               "增强强度：中（从场景beat到完整场景）\n长度目标：2.0x~3.5x\n"
               "重点：节奏铺排、心理层次、环境渲染、微动作衔接\n"
               "限制：严格保留骨架中的全部事实、人名、地名、术语、事件顺序与结果。不新增原文没有的信息。\n\n"
               "素材：\n{text}"),
    "正文润色": "任务：正文润色\n要求：提升文笔、优化节奏、保留原意\n\n原文：\n{text}",
}
WEBNOVEL_DEFAULT_TASK = "骨架扩写"

# 模型②：作者训练用的结构化输入骨架
XUANHUAN_SKELETON = "设定/时代：\n人物状态：\n前情摘要：\n场景目标：\n情节beat：\n冲突：\n结尾状态："

def _length_goal(s):
    return s + f"\n\n（篇幅目标：正文约{CONFIG['target_chars']}个中文字；写完自然收束，不要凑字数，不要输出任何解释性文字。）"

def build_user_message(model_key, text, task=None):
    """把你粘贴的剧情，按所选模型作者推荐的输入模板包装成完整提示词。"""
    text = (text or "").strip()
    if not text:
        raise ValueError("输入为空：请先把剧情粘贴到输入框里。")
    if model_key == "webnovel":
        task = task or WEBNOVEL_DEFAULT_TASK
        return _length_goal(WEBNOVEL_TASKS[task].format(text=text))
    # 模型②：若粘贴内容已带「设定/人物状态/…：」这类标签则原样使用，
    # 否则视为情节beat，自动套进作者的结构化骨架（其余项留给模型合理补全）
    if re.search(r"(设定|人物状态|前情|场景目标|情节beat|冲突|结尾状态)\s*[:：]", text):
        return _length_goal(text)
    filled = ("设定/时代：（由情节自行合理设定）\n人物状态：（由情节自行合理设定）\n前情摘要：（无）\n"
              "场景目标：把情节beat写成完整场景\n情节beat：\n" + text + "\n"
              "冲突：（由情节自行提取）\n结尾状态：（自然收束）")
    return _length_goal(filled)

def clean_output(text):
    """去掉思考模式标签（模型②训练时关闭思考，这里双保险）与首尾空白。
       未闭合的 <think> 视为生成中断在思考段，只保留标签之前的内容。"""
    text = text or ""
    if "<think>" in text:
        if "</think>" not in text:
            text = text.split("<think>")[0]
        else:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
            text = text.replace("<think>", "").replace("</think>", "")
    return text.strip()

# ---------- 批量盲评模式的默认提示词（prompts/ 为空时自动生成） ----------
DEFAULT_PROMPTS = {
    "01_寒潭.txt": ("宗门后山有一口百年寒潭，潭底沉着一块黑铁。杂役弟子沈砚每日来此挑水。\n"
                 "这夜他失足落水，寒气入体本该致命，怀中却发热——那块黑铁竟顺水流进他掌心。\n"
                 "沈砚在潭底憋了半炷香才爬上岸，没死，反而觉得四肢百骸有暖流游走。\n"
                 "翌日晨课，他一拳打裂了练功的石桩。执法长老盯住他手背浮现的黑色纹路。"),
    "02_战斗.txt": ("坊市拍卖会上，沈砚看中的淬体丹被内门弟子赵崂抬价截走。\n"
                 "赵崂当众羞辱杂役也配竞价，动手将沈砚掼倒在地。\n"
                 "沈砚掌心黑铁发烫，气血翻涌，他借势起身，一记崩拳打断赵崂肋骨。\n"
                 "赵崂的随从拔刀围上。拍卖师敲锤喝止，宣布坊市内禁斗，违者逐出。\n"
                 "赵崂撂话三日后演武场见生死状。人群散去，沈砚盯着自己发颤的拳头。"),
    "03_突破.txt": ("沈砚闭关第七日，气海内的暖流凝成一线，卡在炼气九层关口。\n"
                 "黑铁传音：欲破境，需以自身气血温养铁中残魂三成。\n"
                 "沈砚咬牙引气血入铁，如割肉饲鹰，几次险些昏迷。\n"
                 "第三夜铁中残魂反哺一缕先天罡气，气海关口应声而碎。\n"
                 "破入筑基，他吐出一口黑血，发现地上血迹里混着细小铁屑。"),
}

def ensure_prompts_dir():
    d = CONFIG["prompts_dir"]
    os.makedirs(d, exist_ok=True)
    files = sorted(glob.glob(os.path.join(d, "*.txt")))
    if not files:
        for name, body in DEFAULT_PROMPTS.items():
            with open(os.path.join(d, name), "w", encoding="utf-8") as f:
                f.write(body)
        print(f"📝 已生成 {len(DEFAULT_PROMPTS)} 个示例提示词到 {d}/（可自行替换/增删 .txt 文件）")
        files = sorted(glob.glob(os.path.join(d, "*.txt")))
    return files

print("✅ 工具函数就绪（下载/续传/镜像切换/显存监控/自动保存/提示词模板）")

## 第 5 步 · 选择模型并加载（只下载你选中的那一个）

- **「运行全部」时会自动加载默认模型①**（首次下载约 16.5GB，10~40 分钟；之后从持久盘秒加载）；
- 想换模型：下拉框选 → 点【加载 / 切换模型】。选模型②首次会另下约 23.4GB；
- 加载模型②前会先安装深度学习依赖（约 3~6 分钟），并按预案做 A10 显存尝试，**失败会直接给结论，不用你排查**。

In [ ]:
# ============================================================
# 第 5 步：模型加载（llama.cpp / Transformers 双路线 + A10 预案）
# ============================================================
import os, sys, time, subprocess, gc, zipfile, tempfile

# ---------- llama.cpp 运行时（模型①专用）：预编译优先，源码编译兜底 ----------
def ensure_llama_cpp():
    """按优先级准备 llama.cpp：
       ① CUDA 预编译 pip 轮子（约1~2分钟，可流式输出）
       ② llama.cpp 官方预编译二进制（含 CUDA 的 Ubuntu 包）
       ③ 源码编译（首次约15~30分钟，仅需一次）"""
    try:
        import llama_cpp
        print(f"✅ llama-cpp-python 已就绪（{getattr(llama_cpp, '__version__', '?')}）")
        return "python"
    except ImportError:
        pass
    # ① 按本机 CUDA 版本依次尝试预编译轮子
    v = _run("nvidia-smi | grep -o 'CUDA Version: [0-9.]*'")
    m = re.search(r"([0-9]+)\.([0-9]+)", v or "")
    cus = []
    if m:
        cus.append(f"cu{m.group(1)}{m.group(2)}")
    cus += ["cu126", "cu125", "cu124", "cu121"]
    for cu in dict.fromkeys(cus):
        print(f"⏳ 安装 llama-cpp-python（{cu} CUDA 预编译版，约1~2分钟）…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{cu}"],
                       capture_output=True, text=True)
        try:
            import llama_cpp
            print(f"✅ 安装成功（{cu}）")
            return "python"
        except ImportError:
            continue
    # ② 官方预编译二进制
    try:
        import requests
        rel = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest", timeout=30).json()
        assets = [a for a in rel.get("assets", []) if "ubuntu" in a["name"].lower() and a["name"].endswith(".zip")]
        cuda_assets = [a for a in assets if "cuda" in a["name"].lower() or "cublas" in a["name"].lower()]
        pick = (cuda_assets or assets)[0]
        print(f"⏳ pip 轮子不可用，下载官方预编译二进制：{pick['name']} …")
        import urllib.request
        zpath = os.path.join(CONFIG["models_root"], pick["name"])
        urllib.request.urlretrieve(pick["browser_download_url"], zpath)
        bdir = os.path.join(CONFIG["models_root"], "llama-bin")
        with zipfile.ZipFile(zpath) as z:
            z.extractall(bdir)
        cli = glob.glob(os.path.join(bdir, "**", "llama-cli*"), recursive=True)
        STATE["llama_bin"] = cli[0] if cli else None
        if STATE.get("llama_bin"):
            print(f"✅ 官方二进制就绪：{STATE['llama_bin']}（此路线为非流式输出）")
            return "binary"
    except Exception as e:
        print(f"   · 官方二进制不可用（{str(e)[:120]}），转为源码编译…")
    # ③ 源码编译（最后手段）
    print("⏳ 源码编译 llama.cpp（首次约 15~30 分钟，仅需一次，请耐心等待）…")
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on", FORCE_CMAKE="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "llama-cpp-python"],
                       env=env)
    if r.returncode != 0:
        raise RuntimeError("llama.cpp 三种安装方式都失败了，请截图上面的报错信息求助。")
    print("✅ 源码编译完成")
    return "python"

def _is_oom(e):
    s = (type(e).__name__ + " " + str(e)).lower()
    return "outofmemory" in s or "out of memory" in s or "cuda oom" in s

# ---------- 模型①加载 ----------
def load_webnovel():
    cfg = CONFIG["models"]["webnovel"]
    print(f"📥 模型①：检查/下载 {cfg['repo_id']}（{cfg['quant']}，约16.5GB；下载过则直接复用）")
    gguf_path = smart_download(cfg["repo_id"], cfg["gguf_file"])
    runner = ensure_llama_cpp()
    if runner == "python":
        from llama_cpp import Llama
        before = _run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").strip()
        print("⏳ 加载到 GPU（约1~3分钟）…")
        llm = Llama(model_path=gguf_path, n_ctx=cfg["context"], n_gpu_layers=-1,
                    n_threads=max((os.cpu_count() or 8) // 2, 2), verbose=False)
        after = _run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").strip()
        try:
            delta = float(after) - float(before)
        except Exception:
            delta = 0
        if delta < 4000:   # MiB：显存没涨说明没进GPU，会极慢
            print("⚠️ 检测到模型可能没有加载进 GPU（显存占用未上升）。可继续，但生成速度会很慢。")
        STATE.update(loaded_model="webnovel", runner="python", llm=llm, ctx=cfg["context"], offloaded=False)
    else:
        STATE.update(loaded_model="webnovel", runner="binary", llm=gguf_path, ctx=cfg["context"], offloaded=False)
    show_gpu("模型①加载后")
    print("✅ 模型①就绪！请到第 6 步粘贴剧情、点【开始生成】。")

# ---------- 模型②加载（A10 尝试性支持 + 止损预案） ----------
def _xuanhuan_give_up(reason):
    raise SystemExit(
        "🔴 " + reason + "\n"
        "   结论：A10 24GB 不足以稳定运行模型②（4bit 底模 22.3GB 已接近整卡显存）。\n"
        "   这是模型体积问题，不是你的操作错误，也无需继续调整本 Notebook。\n"
        "   建议：改用 80GB 显存的 GPU 实例测试模型②。模型①不受任何影响，可继续使用。")

def load_xuanhuan():
    cfg = CONFIG["models"]["xuanhuan"]
    print("ℹ️  模型②在 A10 24GB 上为【尝试性支持】：4bit 底模约 22.3GB ≈ 整卡显存的 93%。")
    print("    预案：① 纯 GPU 加载 → ② 失败则做唯一一次降级尝试（上下文2048+部分层放CPU）")
    print("          → ③ 仍失败或速度过慢则立即停止并建议 80GB GPU（不做复杂显存优化）。\n")
    pip_install(["transformers", "peft", "accelerate", "bitsandbytes"])
    print("📥 下载 4bit 底模（约22.4GB）+ LoRA 适配器（约1GB）…")
    base_dir = smart_download(cfg["base_repo"])
    adapter_dir = smart_download(cfg["adapter_repo"])
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("⏳ 第 1 次尝试：纯 GPU 加载（上下文 4096，约3~8分钟）…")
    offloaded, eff_ctx = False, cfg["context"]
    tok = AutoTokenizer.from_pretrained(base_dir)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            base_dir, device_map="auto", load_in_4bit=True, torch_dtype=torch.bfloat16)
    except Exception as e:
        if not _is_oom(e):
            raise RuntimeError(f"模型②加载失败（非显存问题），请截图求助：{type(e).__name__}: {str(e)[:300]}")
        print("   · 显存不足（OOM）。执行预案中唯一一次降级尝试：上下文2048 + 限制GPU占用20GiB、其余层放CPU…")
        gc.collect(); torch.cuda.empty_cache()
        try:
            # 降上下文只为生成阶段的 KV 缓存留余量；权重体积不变（这正是 A10 吃紧的根因）
            model = AutoModelForCausalLM.from_pretrained(
                base_dir, device_map="auto", load_in_4bit=True, torch_dtype=torch.bfloat16,
                max_memory={0: f"{cfg['offload_gib']}GiB", "cpu": "60GiB"})
            offloaded, eff_ctx = True, cfg["context_fallback"]
        except Exception as e2:
            if _is_oom(e2):
                _xuanhuan_give_up("两次加载尝试均显存不足。")
            raise RuntimeError(f"模型②加载失败，请截图求助：{type(e2).__name__}: {str(e2)[:300]}")
    model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    show_gpu("模型②加载后")
    # 速度探针：实测生成速度，低于阈值直接止损（避免"能跑但一篇正文等半小时"）
    print("⏳ 速度自检（生成约30 token 实测）…")
    try:
        t0 = time.time()
        _p = tok("你好，", return_tensors="pt").to(model.device)
        with torch.no_grad():
            model.generate(**_p, max_new_tokens=8, do_sample=False, use_cache=True)   # 预热
            t0 = time.time()
            model.generate(**_p, max_new_tokens=24, do_sample=False, use_cache=True)
        tps = 24 / max(time.time() - t0, 1e-6)
        print(f"   实测速度：{tps:.1f} tokens/s" + ("（含CPU卸载）" if offloaded else ""))
        if tps < cfg["min_speed_tps"]:
            _xuanhuan_give_up(f"实测速度 {tps:.1f} tokens/s，低于可用阈值 {cfg['min_speed_tps']}。")
    except Exception as e:
        if _is_oom(e):
            _xuanhuan_give_up("加载成功但生成时显存不足。")
        raise
    STATE.update(loaded_model="xuanhuan", runner="transformers", model=model,
                 tokenizer=tok, ctx=eff_ctx, offloaded=offloaded)
    if offloaded:
        print(f"⚠️ 模型②以降级模式运行（上下文{eff_ctx}，部分层在CPU）。能用，但生成偏慢。")
    print("✅ 模型②就绪！请到第 6 步粘贴剧情、点【开始生成】。")

# ---------- 统一的加载/卸载入口 ----------
def unload_current():
    if STATE.get("loaded_model") is None:
        return
    for k in ("llm", "model", "tokenizer"):
        STATE.pop(k, None)
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    STATE.update(loaded_model=None, runner=None)
    print("🧹 已释放上一个模型的显存")

def load_model(key):
    if STATE.get("loaded_model") == key:
        print("✅ 该模型已在内存中，无需重复加载。")
        return
    unload_current()
    if CONFIG["models"][key]["type"] == "gguf":
        load_webnovel()
    else:
        load_xuanhuan()

# ---------- 交互控件 ----------
import ipywidgets as widgets
from IPython.display import display, clear_output

_sel = widgets.Dropdown(options=[(v["label"], k) for k, v in CONFIG["models"].items()],
                        value=CONFIG["default_model"], description="选择模型：",
                        layout=widgets.Layout(width="auto"))
_btn = widgets.Button(description="加载 / 切换模型", button_style="primary")
_out = widgets.Output()
def _on_load(b):
    with _out:
        clear_output()
        try:
            load_model(_sel.value)
        except SystemExit as e:
            print(str(e))
        except Exception as e:
            print(f"🔴 加载失败：{type(e).__name__}: {str(e)[:300]}\n   常见原因：网络中断（重跑可续传）/ 显存被其他程序占用（重启实例后再试）。")
_btn.on_click(_on_load)
display(widgets.VBox([widgets.HBox([_sel, _btn]), _out]))

# 「运行全部」时自动加载默认模型①，首次体验零点击；想换模型用上面的下拉框
load_model(CONFIG["default_model"])

## 第 6 步 · 粘贴剧情提示词 → 生成正文

在下面的输入框里粘贴你的剧情（骨架、场景描述或待润色正文均可），选好任务类型，点【✍️ 开始生成】。

- 正文会**逐字流式显示**，约 1~5 分钟；
- 生成完**自动保存**到 `results/` 文件夹（文件名含模型名、时间、测试编号）；
- 想再测一次：改输入框内容，再点一次按钮即可，不用重新加载模型。

In [ ]:
# ============================================================
# 第 6 步：生成界面（输入框 + 任务类型 + 流式输出 + 自动保存）
# ============================================================
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

def _default_input(key):
    if key == "webnovel":
        return ("（示例，请替换成你的剧情）\n"
                "少年在废弃藏经阁扫地时捡到半页残缺剑谱，当夜梦见剑招自行演练。\n"
                "醒来后他试着比划，被巡夜长老撞见。")
    return XUANHUAN_SKELETON + "\n（按行填写，冒号后填内容；没有的项可留空。也可以直接粘贴纯剧情，程序会自动套模板）"

def run_generation(model_key, user_msg, system_prompt, cb=None):
    """统一生成入口。cb 为逐段回调（用于流式显示）。返回 dict(text, elapsed)。"""
    if STATE.get("loaded_model") != model_key:
        load_model(model_key)
    t0 = time.time()
    cfg = CONFIG["models"][model_key]
    if model_key == "webnovel":
        text = _gen_webnovel(system_prompt, user_msg, cb)
    else:
        text = _gen_xuanhuan(system_prompt, user_msg, cb)
    return {"text": text, "elapsed": time.time() - t0}

def _gen_webnovel(sys_p, user_p, cb):
    cfg = CONFIG["models"]["webnovel"]
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
    if STATE["runner"] == "python":
        try:
            stream = STATE["llm"].create_chat_completion(
                msgs, temperature=cfg["temperature"], top_p=cfg["top_p"],
                repeat_penalty=cfg["repeat_penalty"], max_tokens=cfg["max_new_tokens"],
                stream=True, jinja=True)   # jinja=True：使用 GGUF 内置的官方对话模板
            parts = []
            for ch in stream:
                d = ch["choices"][0]["delta"].get("content", "")
                if d:
                    parts.append(d)
                    if cb: cb(d)
            return clean_output("".join(parts))
        except TypeError:
            pass   # 旧版本不支持 jinja 参数 → 手工套 Qwen 对话格式
        prompt = (f"<|im_start|>system\n{sys_p}<|im_end|>\n"
                  f"<|im_start|>user\n{user_p}<|im_end|>\n<|im_start|>assistant\n")
        stream = STATE["llm"](prompt=prompt, temperature=cfg["temperature"], top_p=cfg["top_p"],
                              repeat_penalty=cfg["repeat_penalty"], max_tokens=cfg["max_new_tokens"],
                              stream=True)
        parts = []
        for ch in stream:
            d = ch["choices"][0].get("text", "")
            if d:
                parts.append(d)
                if cb: cb(d)
        return clean_output("".join(parts))
    # binary 兜底路线：llama-cli 非流式，一次输出全文
    import tempfile, subprocess
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as tf:
        tf.write(f"<|im_start|>system\n{sys_p}<|im_end|>\n<|im_start|>user\n{user_p}<|im_end|>\n<|im_start|>assistant\n")
        pf = tf.name
    cmd = [STATE["llama_bin"], "-m", STATE["llm"], "-f", pf, "-c", str(cfg["context"]),
           "-n", str(cfg["max_new_tokens"]), "-ngl", "-1", "--temp", str(cfg["temperature"]),
           "--top-p", str(cfg["top_p"]), "--top-k", "40", "--repeat-penalty", str(cfg["repeat_penalty"]),
           "--no-display-prompt"]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    return clean_output(r.stdout.strip())

def _gen_xuanhuan(sys_p, user_p, cb):
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread
    cfg = CONFIG["models"]["xuanhuan"]
    tok, model = STATE["tokenizer"], STATE["model"]
    msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": user_p}]
    try:   # 关闭思考模式（训练时即关闭，推理保持一致）
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=not cfg["disable_thinking"])
    except TypeError:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(**inputs, max_new_tokens=cfg["max_new_tokens"], do_sample=True,
              temperature=cfg["temperature"], top_p=cfg["top_p"], top_k=cfg["top_k"],
              repetition_penalty=cfg["repetition_penalty"], use_cache=True, streamer=streamer)

    def _worker():
        with torch.no_grad():
            model.generate(**kw)
    t = Thread(target=_worker)
    t.start()
    parts = []
    for d in streamer:
        if d:
            parts.append(d)
            if cb: cb(d)
    t.join()
    return clean_output("".join(parts))

# ---------- 界面 ----------
_cur = lambda: STATE.get("loaded_model") or CONFIG["default_model"]
_task_w = widgets.Dropdown(options=list(WEBNOVEL_TASKS.keys()), value=WEBNOVEL_DEFAULT_TASK,
                           description="任务类型：", layout=widgets.Layout(width="auto"),
                           tooltip="仅模型①使用：决定用模型卡里的哪种官方模板包装你的剧情")
_text_w = widgets.Textarea(value=_default_input(_cur()), placeholder="把你的剧情粘贴到这里…",
                           layout=widgets.Layout(width="100%", height="240px"))
_tpl_btn = widgets.Button(description="填入该模型推荐模板", button_style="info",
                          tooltip="清空输入框并填入当前模型的推荐输入格式")
_gen_btn = widgets.Button(description="✍️ 开始生成", button_style="success")
_gen_out = widgets.Output()

def _on_tpl(b):
    _text_w.value = _default_input(_cur())
_tpl_btn.on_click(_on_tpl)

def _on_gen(b):
    with _gen_out:
        clear_output()
        key = _cur()
        task = _task_w.value if key == "webnovel" else "结构化大纲→场景"
        sys_p = CONFIG["models"][key]["system_prompt"]
        try:
            user_msg = build_user_message(key, _text_w.value, _task_w.value)
        except ValueError as e:
            print(f"⚠️ {e}")
            return
        print(f"🚀 模型：{SHORT_NAME[key]}｜任务：{task}｜参数：作者官方推荐值（详见保存的 .md 记录）")
        print(f"⏳ 生成中（目标 {CONFIG['target_chars']} 字，预计1~5分钟），正文如下 ↓\n" + "─" * 60)
        t0 = time.time()
        try:
            res = run_generation(key, user_msg, sys_p, cb=lambda d: print(d, end="", flush=True))
            dt = time.time() - t0
            text = res["text"]
            print("\n" + "─" * 60)
            print(f"✍️ 共 {len(text)} 字，用时 {dt:.0f} 秒（{len(text)/max(dt,1):.1f} 字/秒）")
            meta = {"elapsed": f"{dt:.0f}", "speed": f"{len(text)/max(dt,1):.1f} 字/秒",
                    "vram": _run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader").strip(),
                    "context": STATE.get("ctx", CONFIG["models"][key].get("context"))}
            save_result(key, task, _text_w.value, text, meta)
            show_gpu("生成后")
        except KeyboardInterrupt:
            print("\n⏹️ 已手动停止。调整输入后可再次点击生成。")
        except SystemExit as e:
            print(str(e))
        except Exception as e:
            print(f"\n🔴 生成失败：{type(e).__name__}: {str(e)[:300]}")
            print("   排查建议：① 回第 5 步重新加载模型；② 若提示显存不足，模型②在 A10 属尝试性支持，按提示换 80GB 实例即可。")

_gen_btn.on_click(_on_gen)
display(widgets.VBox([_text_w, widgets.HBox([_task_w, _tpl_btn, _gen_btn]), _gen_out]))
print("👆 在输入框粘贴剧情（或点【填入该模型推荐模板】），然后点【开始生成】。结果自动保存到 results/ 文件夹。")

## 第 7 步 · 换另一个模型再测（对比）

直接回到**第 5 步**：下拉框选另一个模型 → 点【加载 / 切换模型】（首次会下载对应模型，选②约 23.4GB）→ 加载完成后回到**第 6 步**粘贴同一段剧情再生成一次。

两个模型的结果都按「模型名_时间_测试编号」存在同一个 `results/` 文件夹里，方便逐条对比。同一剧情建议用同样的粘贴内容，对比才有意义。

## 第 8 步 ·（可选）批量盲评模式

把若干测试剧情放进 Notebook 同级的 `prompts/` 文件夹（一个 `.txt` 一个剧情，示例会在首次运行时自动生成），然后修改下方 `BATCH_MODELS` 列表并运行本 Cell：

- `[]`：不启用（默认，运行全部时不会误触发）；
- `["webnovel"]`：只跑模型①；
- `["webnovel", "xuanhuan"]`：双模型盲评。

输出保存在 `results/anonymous/`，文件名**隐藏模型名**（`sample_001_A.txt`、`sample_001_B.txt`…），A/B 与真实模型的对照表写在同目录的「AAA_评分后再看_对照表.txt」——**评分之前别打开它**，盲评才有效。

In [ ]:
# ============================================================
# 第 8 步：批量盲评模式（读取 prompts/ 文件夹 → 逐个生成 → 匿名保存）
# ============================================================
BATCH_MODELS = []        # ← 启用请改成 ["webnovel"] 或 ["webnovel", "xuanhuan"]

if not BATCH_MODELS:
    print("ℹ️  批量模式未启用。把上方 BATCH_MODELS 改成 [\"webnovel\"] 或 [\"webnovel\",\"xuanhuan\"] 后重新运行本 Cell。")
    print("   提示：prompts/ 文件夹放测试剧情（.txt），首次运行会自动生成 3 个示例。")
else:
    files = ensure_prompts_dir()
    if not files:
        print("🔴 prompts/ 里没有 .txt 文件，请先放入测试剧情。")
    else:
        letters = {k: chr(65 + i) for i, k in enumerate(CONFIG["models"])}   # A=模型① B=模型②
        anon_dir = os.path.join(CONFIG["results_dir"], "anonymous")
        os.makedirs(anon_dir, exist_ok=True)
        run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        key_path = os.path.join(anon_dir, "AAA_评分后再看_对照表.txt")
        mapping_lines = [f"盲评运行 {run_ts} 对照表（评分前请勿查看！）\n"]
        print(f"▶ 批量模式：{len(files)} 个提示词 × {len(BATCH_MODELS)} 个模型 = {len(files)*len(BATCH_MODELS)} 次生成\n")
        for key in BATCH_MODELS:
            load_model(key)
            mapping_lines.append(f"{letters[key]} = {SHORT_NAME[key]}（{_repo_line(key)}）")
            for i, pf in enumerate(files, 1):
                with open(pf, encoding="utf-8") as f:
                    raw = f.read()
                task = WEBNOVEL_DEFAULT_TASK if key == "webnovel" else "结构化大纲→场景"
                user_msg = build_user_message(key, raw, task)
                print(f"  [{SHORT_NAME[key]}] {i}/{len(files)} {os.path.basename(pf)} 生成中…", end=" ", flush=True)
                t0 = time.time()
                res = run_generation(key, user_msg, CONFIG["models"][key]["system_prompt"])
                dt = time.time() - t0
                meta = {"elapsed": f"{dt:.0f}", "speed": f"{len(res['text'])/max(dt,1):.1f} 字/秒",
                        "vram": _run("nvidia-smi --query-gpu=memory.used --format=csv,noheader").strip(),
                        "context": STATE.get("ctx")}
                save_result(key, task, raw, res["text"], meta)
                anon = os.path.join(anon_dir, f"sample_{i:03d}_{letters[key]}.txt")
                with open(anon, "w", encoding="utf-8") as f:
                    f.write(res["text"].strip() + "\n")
                print(f"完成（{dt:.0f}秒）→ {os.path.basename(anon)}")
            unload_current()
        with open(key_path, "a", encoding="utf-8") as f:
            f.write("\n".join(mapping_lines) + "\n\n")
        print(f"\n✅ 批量完成。匿名结果在 {anon_dir}\n   对照表（评分后再看）：{key_path}")

## 显存随时查看（小工具）

任何时候想看显存占用，运行下面这个 Cell 即可。

In [ ]:
# 显存实时查看
show_gpu("实时")

## 🎬 用完之后：关闭 GPU 实例（省时长）

- ModelScope 免费 GPU 时长按**开机时间**计算（普通用户共 36 小时，CPU 实例不限时）；
- 用完请到 ModelScope 网页 →「我的Notebook」→ 对应实例 → **停止**；
- 停止后 `/mnt/workspace` 里的模型和结果**全部保留**（100GB 持久盘），下次启动直接复用、不用重新下载；
- 只有**删除实例**才会清空文件。删除/长期保存前，记得把 `results/` 里的正文下载到本地：在 Notebook 左侧文件列表找到文件 → 右键 → **Download**。

---

**许可与用途提醒（详见仓库 README）**：本项目仅用于个人对比测试。两个模型分别遵循各自模型卡的许可条款（均含"研究/个人使用"限制，且模型②的训练语料含受版权保护文学作品的派生材料）。任何正式或商业使用前，请分别审查两个模型的许可与训练数据权利。